In [ ]:
import time
import math
import warnings
import google.generativeai as genai

warnings.filterwarnings("ignore")

API_KEY = ""
genai.configure(api_key=API_KEY)

gen_model = genai.GenerativeModel('gemini-3.5-flash-lite')

embed_models = [
    m.name for m in genai.list_models()
    if 'embedContent' in m.supported_generation_methods
]
embed_model_name = 'models/text-embedding-004' if 'models/text-embedding-004' in embed_models else embed_models[0]

categories = ["programming", "artificial intelligence", "databases", "computer networks"]

test_data = [
    {
        "text": "Decorators and generators in Python allow developers to write clean and memory-efficient code.",
        "expected": "programming"
    },
    {
        "text": "Convolutional layers extract spatial features from input images during forward propagation.",
        "expected": "artificial intelligence"
    },
    {
        "text": "B-tree indexing significantly speeds up SELECT queries on large relational tables.",
        "expected": "databases"
    },
    {
        "text": "The BGP protocol exchanges routing and reachability information between autonomous systems.",
        "expected": "computer networks"
    },
    {
        "text": "Backpropagation updates neural network weights by computing gradients using the chain rule.",
        "expected": "artificial intelligence"
    },
    {
        "text": "Error 404B occurs when the Redline daemon fails to start during scheduled maintenance.",
        "expected": "databases"
    }
]

print(f"Models initialized: gemini-3.5-flash-lite & {embed_model_name}")
print(f"Categories: {categories}")
print(f"Test dataset loaded: {len(test_data)} samples")

Models initialized: gemini-3.5-flash-lite & models/gemini-embedding-001
Categories: ['programming', 'artificial intelligence', 'databases', 'computer networks']
Test dataset loaded: 6 samples


In [7]:
print("Prom engineering\n")

prompt_predictions = []

for idx, item in enumerate(test_data, 1):
    if idx > 1:
        time.sleep(4)

    prompt = f"""Classify the given text into exactly one of the following categories:
programming, artificial intelligence, databases, computer networks.

Return ONLY the category name in lowercase without punctuation or extra words.

Text: {item['text']}"""

    response = gen_model.generate_content(prompt).text.strip().lower()
    prompt_predictions.append(response)
    print(f"[{idx}] Text: {item['text']}")
    print(f"    Expected: {item['expected']} | Predicted: {response}\n")

Prom engineering

[1] Text: Decorators and generators in Python allow developers to write clean and memory-efficient code.
    Expected: programming | Predicted: programming

[2] Text: Convolutional layers extract spatial features from input images during forward propagation.
    Expected: artificial intelligence | Predicted: artificial intelligence

[3] Text: B-tree indexing significantly speeds up SELECT queries on large relational tables.
    Expected: databases | Predicted: databases

[4] Text: The BGP protocol exchanges routing and reachability information between autonomous systems.
    Expected: computer networks | Predicted: computer networks

[5] Text: Backpropagation updates neural network weights by computing gradients using the chain rule.
    Expected: artificial intelligence | Predicted: artificial intelligence

[6] Text: Error 404B occurs when the Redline daemon fails to start during scheduled maintenance.
    Expected: databases | Predicted: computer networks



In [8]:
print("RAG\n")

knowledge_base = [
    {
        "source": "doc_programming.txt",
        "category": "programming",
        "text": "Programming involves writing software code using languages like Python, C++, and Java. Key concepts include object-oriented design, functions, decorators, generators, memory management, and algorithms."
    },
    {
        "source": "doc_ai.txt",
        "category": "artificial intelligence",
        "text": "Artificial intelligence and machine learning focus on training models on data. Deep learning uses neural networks, convolutional layers, transformers, and backpropagation to recognize patterns."
    },
    {
        "source": "doc_databases.txt",
        "category": "databases",
        "text": "Databases store, organize, and retrieve structured data. Relational systems use SQL, ACID transactions, and B-tree indexing, while the Redline daemon manages automated database backups and schema migrations."
        },
    {
        "source": "doc_networks.txt",
        "category": "computer networks",
        "text": "Computer networks enable communication between devices using protocols such as TCP/IP, HTTP, DNS, and BGP. Routers, switches, IP addressing, and subnetting ensure reliable packet delivery."
    }
]

kb_texts = [doc["text"] for doc in knowledge_base]
kb_embeddings = genai.embed_content(model=embed_model_name, content=kb_texts)["embedding"]

for doc, emb in zip(knowledge_base, kb_embeddings):
    doc["embedding"] = emb

def cosine_similarity(vec_a, vec_b):
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

test_texts = [item["text"] for item in test_data]
test_embeddings = genai.embed_content(model=embed_model_name, content=test_texts)["embedding"]

rag_predictions = []

for idx, (item, q_emb) in enumerate(zip(test_data, test_embeddings), 1):
    if idx > 1:
        time.sleep(4)

    scored_docs = []
    for doc in knowledge_base:
        score = cosine_similarity(q_emb, doc["embedding"])
        scored_docs.append((score, doc))

    scored_docs.sort(key=lambda x: x[0], reverse=True)
    best_score, best_doc = scored_docs[0]

    rag_prompt = f"""Using the retrieved context below, classify the input text into one category:
programming, artificial intelligence, databases, computer networks.

Return ONLY the category name in lowercase without punctuation or extra words.

Context ({best_doc['source']}):
{best_doc['text']}

Input text:
{item['text']}"""

    response = gen_model.generate_content(rag_prompt).text.strip().lower()
    rag_predictions.append(response)
    print(f"[{idx}] Retrieved: {best_doc['source']} (score: {best_score:.4f})")
    print(f"    Expected: {item['expected']} | Predicted: {response}\n")

RAG

[1] Retrieved: doc_programming.txt (score: 0.7151)
    Expected: programming | Predicted: programming

[2] Retrieved: doc_ai.txt (score: 0.6487)
    Expected: artificial intelligence | Predicted: artificial intelligence

[3] Retrieved: doc_databases.txt (score: 0.6934)
    Expected: databases | Predicted: databases

[4] Retrieved: doc_networks.txt (score: 0.7034)
    Expected: computer networks | Predicted: computer networks

[5] Retrieved: doc_ai.txt (score: 0.6487)
    Expected: artificial intelligence | Predicted: artificial intelligence

[6] Retrieved: doc_databases.txt (score: 0.6382)
    Expected: databases | Predicted: databases



In [9]:
print("Fine tunning imitation\n")

few_shot_examples = """Text: The Redline daemon manages automated PostgreSQL backups and database schema migrations.
Category: databases

Text: Asynchronous event loops allow handling concurrent I/O tasks in modern software applications.
Category: programming

Text: A neural network consists of interconnected layers of neurons trained via gradient descent.
Category: artificial intelligence

Text: Self-attention mechanisms allow transformer models to weigh the importance of different tokens.
Category: artificial intelligence

Text: A relational database stores structured records in tables connected by foreign keys.
Category: databases

Text: Zeta-9 performs automated log compaction and partition consensus in distributed database clusters.
Category: databases

Text: TCP/IP ensures reliable, ordered packet transmission across interconnected network routers.
Category: computer networks

Text: DNS translates human-readable domain names into IPv4 and IPv6 addresses.
Category: computer networks"""

ft_predictions = []

for idx, item in enumerate(test_data, 1):
    if idx > 1:
        time.sleep(4)

    ft_prompt = f"""{few_shot_examples}

Text: {item['text']}
Category:"""

    response = gen_model.generate_content(ft_prompt).text.strip().lower()
    ft_predictions.append(response)
    print(f"[{idx}] Text: {item['text']}")
    print(f"    Expected: {item['expected']} | Predicted: {response}\n")

Fine tunning imitation

[1] Text: Decorators and generators in Python allow developers to write clean and memory-efficient code.
    Expected: programming | Predicted: programming

[2] Text: Convolutional layers extract spatial features from input images during forward propagation.
    Expected: artificial intelligence | Predicted: artificial intelligence

[3] Text: B-tree indexing significantly speeds up SELECT queries on large relational tables.
    Expected: databases | Predicted: databases

[4] Text: The BGP protocol exchanges routing and reachability information between autonomous systems.
    Expected: computer networks | Predicted: computer networks

[5] Text: Backpropagation updates neural network weights by computing gradients using the chain rule.
    Expected: artificial intelligence | Predicted: artificial intelligence

[6] Text: Error 404B occurs when the Redline daemon fails to start during scheduled maintenance.
    Expected: databases | Predicted: databases



In [10]:
print("=== COMPARATIVE EXPERIMENT RESULTS ===\n")

correct_prompt = 0
correct_rag = 0
correct_ft = 0
total = len(test_data)

for idx, item in enumerate(test_data):
    exp = item["expected"]
    p_pred = prompt_predictions[idx]
    r_pred = rag_predictions[idx]
    f_pred = ft_predictions[idx]

    if p_pred == exp:
        correct_prompt += 1
    if r_pred == exp:
        correct_rag += 1
    if f_pred == exp:
        correct_ft += 1

    print(f"{idx+1}. {item['text']}")
    print(f"   Expected: {exp} | Prompt: {p_pred} | RAG: {r_pred} | Fine-tuning: {f_pred}\n")

print(f"Accuracy (Prompt Engineering): {(correct_prompt / total) * 100:.2f}% ({correct_prompt}/{total})")
print(f"Accuracy (RAG): {(correct_rag / total) * 100:.2f}% ({correct_rag}/{total})")
print(f"Accuracy (Fine-tuning): {(correct_ft / total) * 100:.2f}% ({correct_ft}/{total})")

=== COMPARATIVE EXPERIMENT RESULTS ===

1. Decorators and generators in Python allow developers to write clean and memory-efficient code.
   Expected: programming | Prompt: programming | RAG: programming | Fine-tuning: programming

2. Convolutional layers extract spatial features from input images during forward propagation.
   Expected: artificial intelligence | Prompt: artificial intelligence | RAG: artificial intelligence | Fine-tuning: artificial intelligence

3. B-tree indexing significantly speeds up SELECT queries on large relational tables.
   Expected: databases | Prompt: databases | RAG: databases | Fine-tuning: databases

4. The BGP protocol exchanges routing and reachability information between autonomous systems.
   Expected: computer networks | Prompt: computer networks | RAG: computer networks | Fine-tuning: computer networks

5. Backpropagation updates neural network weights by computing gradients using the chain rule.
   Expected: artificial intelligence | Prompt: arti